# Module 05: DataLoader - Efficient Data Pipeline for ML Training

Welcome to Module 05! You're about to build the data loading infrastructure that transforms how ML models consume data during training.

In [2]:

# Essential imports for data loading
import random
import sys
import time
from abc import ABC, abstractmethod
from typing import Iterator, List, Tuple

import numpy as np
rng = np.random.default_rng(7)

import sys
sys.path.append(r"D:\DS\Data Science\TinyTorch\projects\my-tinytorch-systems")
# Import real Tensor class from tinytorch package
from tinytorch.foundation.tensor import Tensor

In [8]:
class Dataset(ABC):

    """ 
    Abstract base class for all datasets

    Provides the fundamental interfaces that all datasests must implement
    __len__(): returns the total number of samples
    __getitem__(idx): returns the sample at given index
    """

    @abstractmethod
    def __len__(self) -> int:
        """
        return the total number of sample in the dataset

        This method must be im[plemented by all subclasses to enable 
        len(dataset) calls and batch size calculations
        
        """

        pass

    @abstractmethod
    def __getitem__(self, idx: int):
        
        """
        Return the sample at the given index.

        Args:
            idx: Index of the sample to retrieve (0 <= idx < len(dataset))

        Returns:
            The sample at index idx. Format depends on the dataset implementation.
            Could be (data, label) tuple, single tensor, etc.
        """
        pass
        

### Unit Test: Dataset Abstract Base Class¶
This test validates our Dataset abstract base class is properly defined.

In [13]:
def test_unit_dataset():
    """ Test Dataset abstract base class..."""
    
    print("Unit test : Dataset abstract base class...")

    # Test that Dataset is properly abstract
    try:
        dataset=Dataset()
        assert False, "Should not be able to instantiate abstract base class"
    except TypeError:
        print("Dataset is properly abstract")

    # Test concrete implementation
    class TestDataset(Dataset):

        def __init__(self, size):
            self.size = size

        def __len__(self):
            return self.size
            
        def __getitem__(self, idx):
            return f"item_{idx}"

    dataset = TestDataset(10)
    assert len(dataset) == 10
    assert dataset[0] == "item_0"
    assert dataset[4] == "item_4"

    print("Dataset interface works correctly")

if __name__ == "__main__":
    test_unit_dataset()
        
            


Unit test : Dataset abstract base class...
Dataset is properly abstract
Dataset interface works correctly


##  TensorDataset - When Data Lives in Memory

Now let's implement TensorDataset, the most common dataset type for when your data is already loaded into tensors. This is perfect for datasets like MNIST where you can fit everything in memory.


In [30]:
class TensorDataset:
    """
    Dataset wrapping tensors for supervised learning
    """

    def __init__(self, *tensors):
        """
            Create dataset from multiple tensors

            Args:
                *tensors: Variable numbers of tensor objects
        """
        assert len(tensors) > 0, "must provide at least one tensor"

        # Store all tensors
        self.tensors = tensors

        # Validate all tensors have same first dimension

        first_size = tensors[0].shape[0]
        for i, tensor in enumerate(tensors):
            if tensor.shape[0] != first_size:
                raise ValueError(
                    f"Tensor size mismatch in TensorDataset\n"
                    f"  ❌ Tensor 0 has {first_size} samples, but Tensor {i} has {tensor.shape[0]} samples\n"
                    f"  💡 All tensors must have the same size in their first dimension (the sample dimension)\n"
                    f"  🔧 Check your data: features.shape[0] should equal labels.shape[0]"
                )
        
    def __len__(self) -> int:
        """ Return number of samples (size of first dimension)"""
        return (self.tensors[0].shape[0])

    def __getitem__(self, idx: int) -> Tuple[Tensor, ...]:
        """ Return the tuple of tensor slice at given index"""

        if idx >= len(self) or idx < 0:
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self)}")
        return tuple(Tensor(tensor.data[idx]) for tensor in self.tensors)
        

        

###  Unit Test: TensorDataset

This test validates our TensorDataset implementation works correctly with tensor-based data.

In [31]:
def test_unit_tensordataset():
    """ Test TensorDataset implementation """
    print (" Unit test : TensorDataset...")

    # Test basic funtionality
    features = Tensor([[1,2], [3,4], [5,6]]) # 3 samples, 2 features
    labels = Tensor([0, 1, 0])               # 3 labels

    dataset = TensorDataset(features, labels)

    # Test length
    assert len(dataset) == 3, f"Expected length 3, got {len(dataset)}"

    # Test indexing
    sample = dataset[0]
    assert len(sample) == 2, "Should return tuple with 2 tensors"
    assert np.array_equal(sample[0].data, [1,2]), f"Wrong features : {sample[0].data}"
    assert sample[1].data == 0, f"Wrong features : {sample[1].data}"

    # Test error handling
    try:
        dataset[10] # out of bounds
        assert False, "Should raise IndexError for out of bounds access"
    except IndexError:
        pass

    # Test mismatched tensor sizes
    try:
        bad_features = Tensor([[1,2], [3,4]]) # only 2 samples
        bad_labels = Tensor([0, 1, 0])        # 3 labels - mismatch !
        dataset = TensorDataset(bad_features, bad_labels)
        assert False, "Should raise error for mismatched tensor sizes"
    except ValueError:
        pass

    print("TensorDataset test works correctly !")

if __name__ == "__main__" :
    test_unit_tensordataset()
        
    

 Unit test : TensorDataset...
TensorDataset test works correctly !


## DataLoader - The Batch Factory

Now we implement the DataLoader class with batching and shuffling support.


In [36]:
class DataLoader:
    """ Data loader with batching and shuffling support """

    def __init__(self, dataset: Dataset, batch_size: int, shuffle: bool = False):
        """ Create dataloader for batched iterartion """
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self) -> int:
        """ Return number of batches per epoch """

        # Calculate the number of complete batches
        return (len(self.dataset) + self.batch_size -1) // self.batch_size
        
    def __iter__(self) -> Iterator: 

         """ Return oterator over batches """

         # Create list of indices
         indices = list(range(len(self.dataset)))

         # Shuffle is requested
         if self.shuffle:
             random.shuffle(indices)

         # Yield batches
         for i in range(0, len(indices), self.batch_size):
             batch_indices = indices[i:i + self.batch_size]
             batch = [self.dataset[idx] for idx in batch_indices]

             # _collate_batch : to convert list of tuples to tuple of tensors
             yield self._collate_batch(batch)

    def _collate_batch(self, batch: list[Tuple[Tensor, ...]]) -> Tuple[Tensor, ...]:
        """ Collate individual samples into batch tensors """

        if len(batch) == 0:
            return ()

        # Determine number of tensors per sample
        num_tensors = len(batch[0])

        # Group tensors by position
        batched_tensors = []
        for tensor_idx in range(num_tensors):
            # extract all tensors at this position
            tensor_list = [sample[tensor_idx].data for sample in batch]

            # stack into batch tensor
            batched_data = np.stack(tensor_list, axis=0)
            batched_tensors.append(Tensor(batched_data))

        return tuple(batched_tensors)
            

##  Data Augmentation - Preventing Overfitting Through Variety

Effective techniques for improving model generalization. By applying random transformations during training, we artificially expand the dataset and force the model to learn robust, invariant features.

In [50]:
class RandomHorizontalFlip:
    """ Randomly flip images horizontally with given probability """

    def __init__(self, p=0.5):
        if not 0.0 <= p <= 1:
            raise ValueError(
                f"Invalid flip probability {p}\n"
                f" p must be between 0.0 and 1.0\n"
                f"p is the probability of flipping the image horizontally (p=0.5 means 50% chance)\n"
                f"common values : (p=0.0, never flip), (p=0.5, standard ), (p=0.1, always flip)"
            )
        self.p = p

    def __call__(self,x):
        """
        Apply random horizontal flip to input.
        """
        if np.random.random() < self.p:
            is_tensor = isinstance(x, Tensor)
            data = x.data if is_tensor else x

            # Determine width axis for HW/CHW/HWC
            # Convention (matching _pad_image and RandomCrop): check shape[0]
            if data.ndim == 2:
                # (H,W)
                axis = -1
            elif data.ndim == 3:
                if data.shape[0] <= 4:
                    # Channels-first #(C, H, W) : flip width (last axis)
                    axis = -1
                else:
                    # Channels-last #(H, W, C) : flip width (second-to-last)
                    axis = -2
            else:
                raise ValueError(
                    f"RandomHorizontalFlip requires at least 2D input\n"
                    f"Got {data.ndim}D with shape {data.shape}\n"
                    f"Images need at least height and width dimensions (H, W) to flip horizontally\n"
                    f"Reshape your data x.reshape(height, witdh) or x.reshape(1, height, witdh)"
                )

            flipped = np.flip(data, axis=axis).copy()
            return Tensor(flipped) if is_tensor else flipped
        return x
                    
                
            
        

### Padding an Image for Random Cropping
Before we can randomly crop an image, we need to pad it with zeros on all sides. This creates extra space so that when we crop back to the original size, we get a slightly shifted version of the image.

In [51]:
def _pad_image(data, padding):
    """ 
        Detect image format and apply zero-padding to spatial dimensions only
    """

    if data.ndim == 2:
        # (H, W) format — pad both axes
        return np.pad(data, padding, mode='constant', constant_values=0)
    elif data.ndim == 3:
        if data.shape[0] <= 4:
            # Channels-first: (C, H, W) — pad only H and W
            return np.pad(data,
                          ((0, 0), (padding, padding), (padding, padding)),
                          mode='constant', constant_values=0)
        else:
            # Channels-last: (H, W, C) — pad only H and W
            return np.pad(data,
                          ((padding, padding), (padding, padding), (0, 0)),
                          mode='constant', constant_values=0)
    else:
        raise ValueError(
            f"RandomCrop requires 2D or 3D input\n"
            f"  ❌ Got {data.ndim}D input with shape {data.shape}\n"
            f"  💡 Expected formats: (H, W) for grayscale, (C, H, W) or (H, W, C) for color images\n"
            f"  🔧 Reshape your data:\n"
            f"     - For single grayscale image: x.reshape(height, width)\n"
            f"     - For single color image: x.reshape(channels, height, width)"
        )
    


###  Unit Test: _pad_image

In [52]:
def test_unit_pad_image():
    """
        Test _pad_image helper
    """
    print("Unit test : _pad_image ...")

    # Test 2D (H, W) - grayscale
    image_hw = np.ones((8,8))
    padded_hw = _pad_image(image_hw, padding=3)
    result_shape = (14,14)
    assert padded_hw.shape == result_shape, f"2D pad shape wrong: {padded_hw.shape}"
    # Corners should be zeros(padding), centers should be ones
    assert padded_hw[0, 0] == 0.0, "Padding should be zero"
    assert padded_hw[4, 4] == 1.0, "Original data should be preserved"

    # Test 3D channels-first (C, H, W)
    image_chw = np.ones((3, 8, 8))
    padded_chw = _pad_image(image_chw, padding=3)
    result_shape = (3, 14, 14)
    assert padded_chw.shape == result_shape, f"3D pad shape wrong: {padded_chw.shape}"
    # Corners should be zeros(padding), centers should be ones
    assert padded_chw[0, 0, 0] == 0.0, "Padding should be zero"
    assert padded_chw[0, 4, 4] == 1.0, "Original data should be preserved"

    # Test 3D channels-last (H, W, C) — shape[0] > 4 triggers this path
    img_hwc = np.ones((32, 32, 3))
    padded_hwc = _pad_image(img_hwc, padding=4)
    assert padded_hwc.shape == (40, 40, 3), f"HWC pad shape wrong: {padded_hwc.shape}"

    try:
        _pad_image(np.ones((10,)), padding=4)
        assert False, "Should raise ValueError for 1D input"
    except ValueError:
        pass

    print("✅ _pad_image works correctly!")

if __name__ == "__main__":
    test_unit_pad_image()

    

Unit test : _pad_image ...
✅ _pad_image works correctly!


### Sampling a Random Crop Region

Once the image is padded, we need to pick a random top-left corner for the crop.
The valid range depends on the padded size minus the target crop size:

This is a pure random sampling operation — no data manipulation, just computing two random integers within valid bounds.

In [53]:
def _random_crop_region(padded_h, padded_w, target_h, target_w):
    """
    Sample a random (top, left) position for cropping.
    """

    top = rng.integers(0, padded_h - target_h + 1)
    left = rng.integers(0, padded_w - target_w + 1)
    return top, left

### Unit Test: _random_crop_region

In [54]:
def test_unit_random_crop_region():
    """ Test random crop region helper """
    print("Unit test: _random_crop_region ...")

    padded_h, padded_w = 40, 40
    target_h, target_w = 32, 32
    max_top = padded_h - target_h   # 8
    max_left = padded_w - target_w  # 8

    # Run many trials and verify bounds
    for _ in range(200):
        top, left = _random_crop_region(padded_h, padded_w, target_h, target_w)
        assert 0 <= top <= max_top, f"top={top} out of range [0, {max_top}]"
        assert 0 <= left <= max_left, f"left={left} out of range [0, {max_left}]"

    # Veriy both endpoints are reachable (Statistical test)
    tops = set()
    lefts = set()
    for _ in range(500):
        t, l = _random_crop_region(padded_h, padded_w, target_h, target_w)
        tops.add(t)
        lefts.add(l)

    assert 0 in tops, "Position 0 should be reachable"
    assert max_top in tops, f"Position {max_top} should be reachable"
    assert 0 in lefts, "Position 0 should be reachable"
    assert max_left in lefts, f"Position {max_left} should be reachable"

    # Edge case: when padded == target, only one valid position
    top, left = _random_crop_region(32, 32, 32, 32)
    assert top == 0 and left == 0, "Only valid position when padded == target"

    print("✅ _random_crop_region works correctly!")

if __name__ == "__main__":
    test_unit_random_crop_region()


    

Unit test: _random_crop_region ...
✅ _random_crop_region works correctly!


## RandomCrop — Composing Pad, Sample, and Extract
Now we combine our two helpers into the complete RandomCrop transform. The  `__call__` method simply orchestrates three clear steps:

In [55]:
class RandomCrop:
    """ Randomly crop image after padding """

    def __init__(self, size, padding=4):
        """ Initialize RandomCrop """

        if isinstance(size, int):
            self.size = (size, size)
        else:
            self.size = size
        self.padding = padding

    def __call__(self, x):
        """
            Apply random crop after padding
        """
        is_tensor = isinstance(x, Tensor)
        data = x.data if is_tensor else x

        target_h, target_w = self.size

        # step 1 : pad the image (handles format detection internally)
        padded = _pad_image(data, self.padding)

        # Step 2 : Determine padded spatial dims and sample crop position
        if data.ndim == 2:
            padded_h, padded_w = padded.shape
            top, left = _random_crop_region(padded_h, padded_w, target_h, target_w)
            cropped = padded[top:top + target_h, left:left + target_w]
        elif data.shape[0] <= 4:
            # Channels-first: (C, H, W)
            padded_h, padded_w = padded.shape[1], padded.shape[2]
            top, left = _random_crop_region(padded_h, padded_w, target_h, target_w)
            cropped = padded[:, top:top + target_h, left:left + target_w]
        else:
            # Channels-last: (H, W, C)
            padded_h, padded_w = padded.shape[0], padded.shape[1]
            top, left = _random_crop_region(padded_h, padded_w, target_h, target_w)
            cropped = padded[top:top + target_h, left:left + target_w, :]

        return Tensor(cropped) if is_tensor else cropped

class Compose:
    """ 
        Compose multiple transforms into a pipeline
        
        Applies transforms in sequence, passing output of each
        as input to the next.

        Args:
        transforms: List of transform callables
    """

    def __init__(self, transforms):
        """
        Initialize Compose with list of transforms

        EXAMPLE:
        >>> transforms = Compose([
        ...     RandomHorizontalFlip(0.5),
        ...     RandomCrop(32, padding=4)
        ... ])
        """

        self.transforms = transforms

    def __call__(self, x):
        """ Apply all transforms in sequence """
        for transform in self.transforms:
            x = transform(x)
        return x

### Unit Test: Data Augmentation Transforms
This test validates our augmentation implementations.

In [56]:
def test_unit_augmentation():

    """ Test data augmentation transforms """
    print("Unit Test: Data Augmentation ...")

    print("Testing RandomHorizontalFlip...")
    flip = RandomHorizontalFlip(p=1.0) # Always flip. For deterministic test

    img = np.array([[1, 2, 3], [4, 5, 6]]) # 2x3 image
    flipped = flip(img)
    expected = np.array([[3, 2, 1], [6, 5, 4]])
    assert np.array_equal(flipped, expected), f"Flip failed: {flipped} vs {expected}"

    # Test never flip
    no_flip = RandomHorizontalFlip(p=0.0)
    unchanged = no_flip(img)
    assert np.array_equal(unchanged, img), "p=0 should never flip"

    # Test RandomCrop shape preservation
    print("  Testing RandomCrop...")
    crop = RandomCrop(32, padding=4)

    # Test with (C, H, W) format (CIFAR-10 style)
    img_chw = rng.standard_normal((3, 32, 32))
    cropped = crop(img_chw)
    assert cropped.shape == (3, 32, 32), f"CHW crop shape wrong: {cropped.shape}"

     # Test with (H, W) format
    img_hw = rng.standard_normal((28, 28))
    crop_hw = RandomCrop(28, padding=4)
    cropped_hw = crop_hw(img_hw)
    assert cropped_hw.shape == (28, 28), f"HW crop shape wrong: {cropped_hw.shape}"

    # Test 3: Compose pipeline
    print("  Testing Compose...")
    transforms = Compose([
        RandomHorizontalFlip(p=0.5),
        RandomCrop(32, padding=4)
    ])

    img = rng.standard_normal((3, 32, 32))
    augmented = transforms(img)
    assert augmented.shape == (3, 32, 32), f"Compose output shape wrong: {augmented.shape}"

    # Test 4: Transforms work with Tensor
    print("  Testing Tensor compatibility...")
    tensor_img = Tensor(rng.standard_normal((3, 32, 32)))

    flip_result = RandomHorizontalFlip(p=1.0)(tensor_img)
    assert isinstance(flip_result, Tensor), "Flip should return Tensor when given Tensor"

    crop_result = RandomCrop(32, padding=4)(tensor_img)
    assert isinstance(crop_result, Tensor), "Crop should return Tensor when given Tensor"

    # Test 5: Randomness verification
    print("  Testing randomness...")
    flip_random = RandomHorizontalFlip(p=0.5)

    # Run many times and check we get both outcomes
    flips = 0
    no_flips = 0
    test_img = np.array([[1, 2]])

    for _ in range(100):
        result = flip_random(test_img)
        if np.array_equal(result, np.array([[2, 1]])):
            flips += 1
        else:
            no_flips += 1

    # With p=0.5, we should get roughly 50/50 (allow for randomness)
    assert flips > 20 and no_flips > 20, f"Flip randomness seems broken: {flips} flips, {no_flips} no-flips"

    print("✅ Data Augmentation works correctly!")

if __name__ == "__main__":
    test_unit_augmentation()
    

Unit Test: Data Augmentation ...
Testing RandomHorizontalFlip...
  Testing RandomCrop...
  Testing Compose...
  Testing Tensor compatibility...
  Testing randomness...
✅ Data Augmentation works correctly!


### Unit Test: DataLoader

This test validates our DataLoader implementation with batching and shuffling.

In [70]:
def test_unit_dataloader():
    """ Test DataLoader implementation."""
    print(" Unit Test: DataLoader...")

    # Create test dataset
    features = Tensor([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]])  # 5 samples
    labels = Tensor([0, 1, 0, 1, 0])
    dataset = TensorDataset(features, labels)

    # Test basic batching (no shuffle)
    loader = DataLoader(dataset, batch_size=2, shuffle=False)

    # Test length calculation
    assert len(loader) == 3, f"Expected 3 batches, got {len(loader)}"  # ceil(5/2) = 3

    batches = list(loader)
    assert len(batches) == 3, f"Expected 3 batches, got {len(batches)}"

    # Test first batch
    batch_features, batch_labels = batches[0]
    assert batch_features.data.shape == (2, 2), f"Wrong batch features shape: {batch_features.data.shape}"
    assert batch_labels.data.shape == (2,), f"Wrong batch labels shape: {batch_labels.data.shape}"

    # Test last batch (should have 1 sample)
    batch_features, batch_labels = batches[2]
    assert batch_features.data.shape == (1, 2), f"Wrong last batch features shape: {batch_features.data.shape}"
    assert batch_labels.data.shape == (1,), f"Wrong last batch labels shape: {batch_labels.data.shape}"

    # Test that data is preserved
    assert np.array_equal(batches[0][0].data[0], [1, 2]), "First sample should be [1,2]"
    assert batches[0][1].data[0] == 0, "First label should be 0"

    # Test shuffling produces different order
    loader_shuffle = DataLoader(dataset, batch_size=5, shuffle=True)
    loader_no_shuffle = DataLoader(dataset, batch_size=5, shuffle=False)
    
    batch_shuffle = list(loader_shuffle)[0]
    batch_no_shuffle = list(loader_no_shuffle)[0]

    # Note: This might occasionally fail due to random chance, but very unlikely
    # We'll just test that both contain all the original data
    shuffle_features = set(tuple(row) for row in batch_shuffle[0].data)
    no_shuffle_features = set(tuple(row) for row in batch_no_shuffle[0].data)
    expected_features = {(1, 2), (3, 4), (5, 6), (7, 8), (9, 10)}

    assert shuffle_features == expected_features, "Shuffle should preserve all data"
    assert no_shuffle_features == expected_features, "No shuffle should preserve all data"

    print("✅ DataLoader works correctly!")

if __name__ == "__main__":
    test_unit_dataloader()

 Unit Test: DataLoader...
✅ DataLoader works correctly!


### Unit Test: DataLoader Deterministic Shuffling
This test validates deterministic shuffling with fixed random seeds.

In [73]:
def test_unit_dataloader_deterministic():
    """ Test DataLoader deterministic shuffling with fixed seed."""
    print(" Unit Test: DataLoader Deterministic Shuffling...")

    # Create test dataset
    features = Tensor([[1, 2], [3, 4], [5, 6], [7, 8]])
    labels = Tensor([0, 1, 0, 1])
    dataset = TensorDataset(features, labels)

    # Test that same seed produces same shuffle
    random.seed(42)
    loader1 = DataLoader(dataset, batch_size=2, shuffle=True)
    batches1 = list(loader1)
    batches2 = len(loader1)
    
    
    random.seed(42)
    loader2 = DataLoader(dataset, batch_size=2, shuffle=True)
    batches2 = list(loader2)

    # Should produce identical batches with same seed
    for i, (batch1, batch2) in enumerate(zip(batches1, batches2)):
        assert np.array_equal(batch1[0].data, batch2[0].data), \
            f"Batch {i} features should be identical with same seed"
        assert np.array_equal(batch1[1].data, batch2[1].data), \
            f"Batch {i} labels should be identical with same seed"

    # Test that different seeds produce different shuffles
    random.seed(42)
    loader3 = DataLoader(dataset, batch_size=2, shuffle=True)
    batches3 = list(loader3)

    random.seed(123)  # Different seed
    loader4 = DataLoader(dataset, batch_size=2, shuffle=True)
    batches4 = list(loader4)

    # Should produce different batches with different seeds (very likely)
    different = False
    for batch3, batch4 in zip(batches3, batches4):
        if not np.array_equal(batch3[0].data, batch4[0].data):
            different = True
            break

    assert different, "Different seeds should produce different shuffles"

    print("✅ Deterministic shuffling works correctly!")

if __name__ == "__main__":
    test_unit_dataloader_deterministic()

 Unit Test: DataLoader Deterministic Shuffling...
Batches :  [(Tensor(data=[[5. 6.]
 [3. 4.]], shape=(2, 2)), Tensor(data=[0. 1.], shape=(2,))), (Tensor(data=[[7. 8.]
 [1. 2.]], shape=(2, 2)), Tensor(data=[1. 0.], shape=(2,)))]
len Batches :  2
✅ Deterministic shuffling works correctly!


## Systems Analysis - Data Pipeline Performance

### Pipeline Bottleneck Identification

We'll measure three critical metrics:

1. **Throughput**: Samples processed per second
2. **Memory Usage**: Peak memory during batch loading
3. **Overhead**: Time spent on data vs computation

These measurements will reveal whether our pipeline is CPU-bound (slow data loading) or compute-bound (slow model).

In [74]:
def analyze_dataloader_performance():
    """ Analyze DataLoader performance characteristics."""
    print(" Analyzing DataLoader Performance...")

    # Create test dataset of varying sizes
    sizes = [1000, 5000, 10000]
    batch_sizes = [16, 64, 256]

    print("\n🔍 Batch Size vs Loading Time:")

    for size in sizes:
        # Create synthetic dataset
        features = Tensor(rng.standard_normal((size, 100)))  # 100 features
        labels = Tensor(rng.integers(0, 10, size))
        dataset = TensorDataset(features, labels)

        print(f"\nDataset size: {size} samples")

        for batch_size in batch_sizes:
            # Time data loading
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

            start_time = time.time()
            batch_count = 0
            for batch in loader:
                batch_count += 1
            end_time = time.time()

            elapsed = end_time - start_time
            throughput = size / elapsed if elapsed > 0 else float('inf')

            print(f"  Batch size {batch_size:3d}: {elapsed:.3f}s ({throughput:,.0f} samples/sec)")

    # Analyze shuffle overhead
    print("\n🔄 Shuffle Overhead Analysis:")

    dataset_size = 10000
    features = Tensor(rng.standard_normal((dataset_size, 50)))
    labels = Tensor(rng.integers(0, 5, dataset_size))
    dataset = TensorDataset(features, labels)

    batch_size = 64

    # No shuffle
    loader_no_shuffle = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    start_time = time.time()
    batches_no_shuffle = list(loader_no_shuffle)
    time_no_shuffle = time.time() - start_time

    # With shuffle
    loader_shuffle = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    start_time = time.time()
    batches_shuffle = list(loader_shuffle)
    time_shuffle = time.time() - start_time

    shuffle_overhead = ((time_shuffle - time_no_shuffle) / time_no_shuffle) * 100

    print(f"  No shuffle: {time_no_shuffle:.3f}s")
    print(f"  With shuffle: {time_shuffle:.3f}s")
    print(f"  Shuffle overhead: {shuffle_overhead:.1f}%")

    print("\n💡 Key Insights:")
    print("• Larger batch sizes reduce per-sample overhead")
    print("• Shuffle adds minimal overhead for reasonable dataset sizes")
    print("• Memory usage scales linearly with batch size")
    print("🚀 Production tip: Balance batch size with GPU memory limits")


def analyze_memory_usage():
    """ Analyze memory usage patterns in data loading."""
    print("\n Analyzing Memory Usage Patterns...")

    # Memory usage estimation
    def estimate_memory_mb(batch_size, feature_size, dtype_bytes=4):
        """Estimate memory usage for a batch."""
        return (batch_size * feature_size * dtype_bytes) / (1024 * 1024)

    print("\n💾 Memory Usage by Batch Configuration:")

    feature_sizes = [784, 3072, 150528]  # MNIST, CIFAR-10, ImageNet-like
    feature_names = ["MNIST (28×28)", "CIFAR-10 (32×32×3)", "ImageNet (224×224×3)"]
    batch_sizes = [1, 32, 128, 512]

    for feature_size, name in zip(feature_sizes, feature_names):
        print(f"\n{name}:")
        for batch_size in batch_sizes:
            memory_mb = estimate_memory_mb(batch_size, feature_size)
            print(f"  Batch {batch_size:3d}: {memory_mb:6.1f} MB")

    print("\n🎯 Memory Trade-offs:")
    print("• Larger batches: More memory, better GPU utilization")
    print("• Smaller batches: Less memory, more noisy gradients")
    print("• Sweet spot: Usually 32-128 depending on model size")

    # Demonstrate actual memory usage with our tensors
    print("\n🧪 Actual Tensor Memory Usage:")

    # Create different sized tensors
    tensor_small = Tensor(rng.standard_normal((32, 784)))    # Small batch
    tensor_large = Tensor(rng.standard_normal((512, 784)))   # Large batch

    # Measure actual memory (data array + object overhead)
    small_bytes = tensor_small.data.nbytes
    large_bytes = tensor_large.data.nbytes

    # Also measure Python object overhead
    small_total = sys.getsizeof(tensor_small.data) + sys.getsizeof(tensor_small)
    large_total = sys.getsizeof(tensor_large.data) + sys.getsizeof(tensor_large)

    print("  Small batch (32×784):")
    print(f"    - Data only: {small_bytes / 1024:.1f} KB")
    print(f"    - With object overhead: {small_total / 1024:.1f} KB")
    print("  Large batch (512×784):")
    print(f"    - Data only: {large_bytes / 1024:.1f} KB")
    print(f"    - With object overhead: {large_total / 1024:.1f} KB")
    print(f"  Ratio: {large_bytes / small_bytes:.1f}× (data scales linearly)")

    print("\n🎯 Memory Optimization Tips:")
    print("• Object overhead becomes negligible with larger batches")
    print("• Use float32 instead of float64 to halve memory usage")
    print("• Consider gradient accumulation for effective larger batches")


def analyze_collation_overhead():
    """📊 Analyze the cost of collating samples into batches."""
    print("\n📊 Analyzing Collation Overhead...")

    # Test different batch sizes to see collation cost
    dataset_size = 1000
    feature_size = 100
    features = Tensor(rng.standard_normal((dataset_size, feature_size)))
    labels = Tensor(rng.integers(0, 10, dataset_size))
    dataset = TensorDataset(features, labels)

    print("\n⚡ Collation Time by Batch Size:")

    for batch_size in [8, 32, 128, 512]:
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        start_time = time.time()
        for batch in loader:
            pass  # Just iterate, measuring collation overhead
        total_time = time.time() - start_time

        batches = len(loader)
        time_per_batch = (total_time / batches) * 1000  # Convert to ms

        print(f"  Batch size {batch_size:3d}: {time_per_batch:.2f}ms per batch ({batches} batches total)")

    print("\n💡 Collation Insights:")
    print("• Larger batches take longer to collate (more np.stack operations)")
    print("• But fewer large batches are more efficient than many small ones")
    print("• Optimal: Balance between batch size and iteration overhead")


# Run the systems analysis (uncomment to run)
if __name__ == "__main__":
    
    analyze_dataloader_performance()
    analyze_memory_usage()
    analyze_collation_overhead()

 Analyzing DataLoader Performance...

🔍 Batch Size vs Loading Time:

Dataset size: 1000 samples
  Batch size  16: 0.013s (76,004 samples/sec)
  Batch size  64: 0.011s (92,465 samples/sec)
  Batch size 256: 0.009s (115,073 samples/sec)

Dataset size: 5000 samples
  Batch size  16: 0.030s (169,214 samples/sec)
  Batch size  64: 0.022s (227,884 samples/sec)
  Batch size 256: 0.024s (208,751 samples/sec)

Dataset size: 10000 samples
  Batch size  16: 0.047s (210,576 samples/sec)
  Batch size  64: 0.053s (188,588 samples/sec)
  Batch size 256: 0.045s (222,048 samples/sec)

🔄 Shuffle Overhead Analysis:
  No shuffle: 0.047s
  With shuffle: 0.049s
  Shuffle overhead: 5.4%

💡 Key Insights:
• Larger batch sizes reduce per-sample overhead
• Shuffle adds minimal overhead for reasonable dataset sizes
• Memory usage scales linearly with batch size
🚀 Production tip: Balance batch size with GPU memory limits

 Analyzing Memory Usage Patterns...

💾 Memory Usage by Batch Configuration:

MNIST (28×28):
 

###  Integration Test: Training Workflow

Let's test how our DataLoader integrates with a complete training workflow, simulating real ML pipeline usage.

In [75]:
def test_unit_training_integration():
    """ Test DataLoader integration with training workflow."""
    print(" Integration Test: Training Workflow...")

    # Create a realistic dataset
    num_samples = 1000
    num_features = 20
    num_classes = 5

    # Synthetic classification data
    features = Tensor(rng.standard_normal((num_samples, num_features)))
    labels = Tensor(rng.integers(0, num_classes, num_samples))

    dataset = TensorDataset(features, labels)

    # Create train/val splits
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    # Manual split (in production, you'd use proper splitting utilities)
    train_indices = list(range(train_size))
    val_indices = list(range(train_size, len(dataset)))

    # Create subset datasets
    train_samples = [dataset[i] for i in train_indices]
    val_samples = [dataset[i] for i in val_indices]

    # Convert back to tensors for TensorDataset
    train_features = Tensor(np.stack([sample[0].data for sample in train_samples]))
    train_labels = Tensor(np.stack([sample[1].data for sample in train_samples]))
    val_features = Tensor(np.stack([sample[0].data for sample in val_samples]))
    val_labels = Tensor(np.stack([sample[1].data for sample in val_samples]))

    train_dataset = TensorDataset(train_features, train_labels)
    val_dataset = TensorDataset(val_features, val_labels)

    # Create DataLoaders
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print("📊 Dataset splits:")
    print(f"  Training: {len(train_dataset)} samples, {len(train_loader)} batches")
    print(f"  Validation: {len(val_dataset)} samples, {len(val_loader)} batches")

    # Simulate training loop
    print("\n Simulated Training Loop:")

    epoch_samples = 0
    batch_count = 0

    for batch_idx, (batch_features, batch_labels) in enumerate(train_loader):
        batch_count += 1
        epoch_samples += len(batch_features.data)

        # Simulate forward pass (just check shapes)
        assert batch_features.data.shape[0] <= batch_size, "Batch size exceeded"
        assert batch_features.data.shape[1] == num_features, "Wrong feature count"
        assert len(batch_labels.data) == len(batch_features.data), "Mismatched batch sizes"

        if batch_idx < 3:  # Show first few batches
            print(f"  Batch {batch_idx + 1}: {batch_features.data.shape[0]} samples")

    print(f"  Total: {batch_count} batches, {epoch_samples} samples processed")

    # Validate that all samples were seen
    assert epoch_samples == len(train_dataset), f"Expected {len(train_dataset)}, processed {epoch_samples}"

    print("✅ Training integration works correctly!")

if __name__ == "__main__":
    test_unit_training_integration()

 Integration Test: Training Workflow...
📊 Dataset splits:
  Training: 800 samples, 25 batches
  Validation: 200 samples, 7 batches

🏃 Simulated Training Loop:
  Batch 1: 32 samples
  Batch 2: 32 samples
  Batch 3: 32 samples
  Total: 25 batches, 800 samples processed
✅ Training integration works correctly!


##  Module Integration Test

Final validation that everything works together correctly before module completion.

In [77]:
def test_module():
    """🧪 Module Test: Complete Integration

    Comprehensive test of entire module functionality.

    This final test runs before module summary to ensure:
    - All unit tests pass
    - Functions work together correctly
    - Module is ready for integration with TinyTorch
    """
    print("🧪 RUNNING MODULE INTEGRATION TEST")
    print("=" * 50)

    # Run all unit tests
    print("Running unit tests...")
    test_unit_dataset()
    test_unit_tensordataset()
    test_unit_dataloader()
    test_unit_dataloader_deterministic()
    test_unit_pad_image()
    test_unit_random_crop_region()
    test_unit_augmentation()

    print("\nRunning integration scenarios...")

    # Test complete workflow
    test_unit_training_integration()

    # Test augmentation with DataLoader
    print("🧪 Integration Test: Augmentation with DataLoader...")

    # Create dataset with augmentation
    train_transforms = Compose([
        RandomHorizontalFlip(0.5),
        RandomCrop(8, padding=2)  # Small images for test
    ])

    # Simulate CIFAR-style images (C, H, W)
    images = rng.standard_normal((100, 3, 8, 8))
    labels = rng.integers(0, 10, 100)

    # Apply augmentation manually (how you'd use in practice)
    augmented_images = np.array([train_transforms(img) for img in images])

    dataset = TensorDataset(Tensor(augmented_images), Tensor(labels))
    loader = DataLoader(dataset, batch_size=16, shuffle=True)

    batch_count = 0
    for batch_x, batch_y in loader:
        assert batch_x.shape[1:] == (3, 8, 8), f"Augmented batch shape wrong: {batch_x.shape}"
        batch_count += 1

    assert batch_count > 0, "DataLoader should produce batches"
    print("✅ Augmentation + DataLoader integration works!")

    print("\n" + "=" * 50)
    print("🎉 ALL TESTS PASSED! Module ready for export.")

In [78]:
def demo_dataloader():
    """🎯 DataLoader batch data correctly."""
    print("🎯 DataLoader Batches Your Data")
    print("=" * 45)

    # Create a dataset
    X = Tensor(rng.standard_normal((100, 64)))
    y = Tensor(np.arange(100))
    dataset = TensorDataset(X, y)

    # Create DataLoader with batching
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    print(f"Dataset: {len(dataset)} samples")
    print("Batch size: 32")
    print(f"Number of batches: {len(loader)}")

    print("\nBatches:")
    for i, (batch_x, batch_y) in enumerate(loader):
        print(f"  Batch {i+1}: {batch_x.shape[0]} samples, shape {batch_x.shape}")

    print("\n✨  DataLoader organizes data for efficient training!")

In [79]:
if __name__ == "__main__":
    test_module()
    print("\n")
    demo_dataloader()

🧪 RUNNING MODULE INTEGRATION TEST
Running unit tests...
Unit test : Dataset abstract base class...
Dataset is properly abstract
Dataset interface works correctly
 Unit test : TensorDataset...
TensorDataset test works correctly !
 Unit Test: DataLoader...
✅ DataLoader works correctly!
 Unit Test: DataLoader Deterministic Shuffling...
Batches :  [(Tensor(data=[[5. 6.]
 [3. 4.]], shape=(2, 2)), Tensor(data=[0. 1.], shape=(2,))), (Tensor(data=[[7. 8.]
 [1. 2.]], shape=(2, 2)), Tensor(data=[1. 0.], shape=(2,)))]
len Batches :  2
✅ Deterministic shuffling works correctly!
Unit test : _pad_image ...
✅ _pad_image works correctly!
Unit test: _random_crop_region ...
✅ _random_crop_region works correctly!
Unit Test: Data Augmentation ...
Testing RandomHorizontalFlip...
  Testing RandomCrop...
  Testing Compose...
  Testing Tensor compatibility...
  Testing randomness...
✅ Data Augmentation works correctly!

Running integration scenarios...
 Integration Test: Training Workflow...
📊 Dataset splits: